# Q-learning on a gridworld: hands-on (optional)
**LLMs From Basics · Lesson RL.3**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (or a ✓) means success.

You'll build the course's 4×4 gridworld from scratch in NumPy, solve it exactly with **value iteration** (RL.2), then learn the same answer from experience with **tabular Q-learning** and ε-greedy exploration, and check that the learned policy matches the optimal one. No `gym` needed.

```
row 0:  .  .  .  G      G = goal, reward +1, episode ends
row 1:  .  #  .  P      P = pit,  reward −1, episode ends;  # = wall
row 2:  .  .  .  .
row 3:  S  .  .  .      S = start
```

Moves are up/down/left/right and deterministic; bumping into the wall or the edge leaves you in place. Every other move gives reward 0, and $\gamma = 0.9$.

Stuck? Solutions are at the very bottom. Only `numpy` and `matplotlib` are needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N_ROWS, N_COLS = 4, 4
GOAL, PIT, WALL, START = (0, 3), (1, 3), (1, 1), (3, 0)
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # up, down, left, right
ARROWS = "↑↓←→"
GAMMA = 0.9

def to_s(rc):  return rc[0] * N_COLS + rc[1]           # (row, col) -> state index 0..15
def to_rc(s):  return divmod(s, N_COLS)                 # state index -> (row, col)
TERMINAL = {to_s(GOAL), to_s(PIT)}
VALID = [s for s in range(16) if to_rc(s) != WALL]      # 15 cells you can stand in
NONTERMINAL = [s for s in VALID if s not in TERMINAL]  # 13 cells where you choose a move

def show_values(V):
    for r in range(N_ROWS):
        row = []
        for c in range(N_COLS):
            s = to_s((r, c))
            row.append("  ## " if (r, c) == WALL else (" +1  " if s == to_s(GOAL) else (" -1  " if s == to_s(PIT) else f"{V[s]:5.2f}")))
        print(" ".join(row))

def show_policy(policy):
    for r in range(N_ROWS):
        row = []
        for c in range(N_COLS):
            s = to_s((r, c))
            row.append("#" if (r, c) == WALL else ("G" if s == to_s(GOAL) else ("P" if s == to_s(PIT) else ARROWS[policy[s]])))
        print(" ".join(row))
print("ready")

## 1 · The environment
Write `step(s, a)` that returns `(next_state, reward, done)` for state index `s` and action index `a` (0 up, 1 down, 2 left, 3 right).

- Moving off the board or into the wall: stay in `s`.
- Landing on the goal: reward `+1.0`, `done=True`. On the pit: reward `-1.0`, `done=True`. Otherwise reward `0.0`, `done=False`.

In [ ]:
def step(s, a):
    r, c = to_rc(s)
    dr, dc = ACTIONS[a]
    # TODO: compute the next cell (stay put at edges and the wall), then the reward and done flag
    raise NotImplementedError("Exercise 1")

**Check 1**

In [ ]:
assert step(to_s((0, 2)), 3) == (to_s(GOAL), 1.0, True), "right from next-to-goal reaches the goal"
assert step(to_s((2, 3)), 0) == (to_s(PIT), -1.0, True), "up from below the pit falls in"
assert step(to_s((1, 0)), 3) == (to_s((1, 0)), 0.0, False), "the wall blocks you"
assert step(to_s((3, 0)), 2) == (to_s((3, 0)), 0.0, False), "the edge blocks you"
assert step(to_s((3, 0)), 0) == (to_s((2, 0)), 0.0, False)
print("✓ exercise 1")

## 2 · Value iteration (the exact answer)
Repeat the Bellman optimality backup for every non-terminal state until nothing changes:

$$V(s) \leftarrow \max_a \big[\, r + \gamma\, V(s') \,\big] \qquad (V(s') = 0 \text{ if } s' \text{ is terminal})$$

Use the *old* values for the whole sweep (copy `V` first). Return the final `V` (length 16; terminal and wall entries stay 0) and the number of sweeps that changed something.

In [ ]:
def value_iteration(gamma=GAMMA, tol=1e-10):
    V = np.zeros(16)
    sweeps = 0
    while True:
        # TODO: one synchronous sweep over NONTERMINAL states; stop when max change < tol
        raise NotImplementedError("Exercise 2")
    return V, sweeps

**Check 2.** These are the lesson's numbers: the start is worth $0.9^5 \approx 0.59$ and it takes 6 sweeps.

In [ ]:
V_star, n_sweeps = value_iteration()
show_values(V_star)
assert abs(V_star[to_s((0, 2))] - 1.0) < 1e-9
assert abs(V_star[to_s((3, 0))] - 0.9**5) < 1e-9
assert abs(V_star[to_s((2, 3))] - 0.729) < 1e-9, "below the pit: walk around, 4 moves"
assert n_sweeps == 6, n_sweeps
print("✓ exercise 2")

The optimal greedy actions (all ties) come from a one-step look-ahead on $V^*$. This helper is given:

In [ ]:
def optimal_actions(V, gamma=GAMMA):
    best = {}
    for s in NONTERMINAL:
        q = []
        for a in range(4):
            s2, r, done = step(s, a)
            q.append(r + (0.0 if done else gamma * V[s2]))
        q = np.array(q)
        best[s] = set(np.flatnonzero(np.isclose(q, q.max())))
    return best

OPT = optimal_actions(V_star)
show_policy({s: min(a) for s, a in OPT.items()})

## 3 · ε-greedy action selection
With probability `eps`, return a uniformly random action. Otherwise return the action with the highest `Q[s]`, **breaking ties at random** (important at the start, when every Q is 0). Use the `rng` you are given.

In [ ]:
def epsilon_greedy(Q, s, eps, rng):
    # TODO: explore with probability eps, else a random choice among the argmax actions
    raise NotImplementedError("Exercise 3")

**Check 3**

In [ ]:
rng = np.random.default_rng(0)
Qt = np.zeros((16, 4)); Qt[4, 2] = 1.0                 # state 4: action 2 is best
picks = [epsilon_greedy(Qt, 4, 0.0, rng) for _ in range(200)]
assert set(picks) == {2}, "eps = 0 must always exploit"
counts = np.bincount([epsilon_greedy(Qt, 4, 1.0, rng) for _ in range(4000)], minlength=4)
assert counts.min() > 850, f"eps = 1 should be roughly uniform: {counts}"
ties = np.bincount([epsilon_greedy(np.zeros((16, 4)), 0, 0.0, rng) for _ in range(4000)], minlength=4)
assert ties.min() > 850, f"ties must be broken at random: {ties}"
print("✓ exercise 3")

## 4 · Tabular Q-learning
For each episode, start at `START` and repeat until `done` (or 200 moves):
1. pick `a = epsilon_greedy(Q, s, eps, rng)`,
2. take `s2, r, done = step(s, a)`,
3. update $Q(s,a) \leftarrow Q(s,a) + \alpha\,[\,r + \gamma \max_{a'} Q(s',a') - Q(s,a)\,]$, with the max term set to 0 when `done`.

Return the Q-table (16×4) and a list with the number of moves in each episode.

In [ ]:
def q_learning(n_episodes=500, alpha=0.5, gamma=GAMMA, eps=0.2, seed=0, max_moves=200):
    rng = np.random.default_rng(seed)
    Q = np.zeros((16, 4))
    lengths = []
    for ep in range(n_episodes):
        s, done, moves = to_s(START), False, 0
        # TODO: run one episode with epsilon-greedy actions and Q-learning updates
        raise NotImplementedError("Exercise 4")
        lengths.append(moves)
    return Q, lengths

**Check 4.** The worked example from the lesson first, then a full run.

In [ ]:
# the lesson's single update: Q = 0.2, r = 0, max Q(s', .) = 0.5, alpha = 0.5
q, target = 0.2, 0 + 0.9 * 0.5
assert abs(q + 0.5 * (target - q) - 0.325) < 1e-12

Q, lengths = q_learning(n_episodes=500, seed=0)
show_values(Q.max(axis=1))
assert abs(Q.max(axis=1)[to_s(START)] - 0.9**5) < 0.05, "max_a Q(start, a) should approach V*(start) = 0.59"
assert np.mean(lengths[-50:]) < 9, "late episodes should be short (6 moves is optimal; ε adds a few)"
print("✓ exercise 4")

## 5 · Did it find the optimal policy?
Write `agreement(Q, opt)` that returns the fraction of non-terminal states where **every** greedy action of `Q` (all ties of `Q[s].max()`) is in the optimal set `opt[s]`.

In [ ]:
def agreement(Q, opt):
    # TODO: fraction of NONTERMINAL states whose greedy action(s) are all optimal
    raise NotImplementedError("Exercise 5")

**Check 5**

In [ ]:
Q, _ = q_learning(n_episodes=2000, seed=1)
frac = agreement(Q, OPT)
print(f"greedy policy is optimal in {frac:.0%} of states")
show_policy({s: int(np.argmax(Q[s])) for s in NONTERMINAL})
assert frac == 1.0
assert agreement(np.zeros((16, 4)), OPT) == 0.0, "an untrained table (all ties) is not optimal anywhere"
print("✓ exercise 5")

## Experiment
1. **Exploration.** Run `q_learning` with `eps` = 0, 0.1 and 0.5 (same number of episodes) and compare the agreement and the episode lengths below. Why can `eps = 0` get stuck, and why is `eps = 0.5` slow even late in training?
2. **Learning rate.** Try `alpha = 0.05` and `alpha = 1.0`. This world is deterministic, so what is special about `alpha = 1`?
3. **A slippery world.** Change `step` so that with probability 0.2 the agent moves in a random direction instead. Rerun value iteration (average over outcomes!) and Q-learning. Does the policy now keep further away from the pit?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for eps in [0.0, 0.1, 0.5]:
    Q, lengths = q_learning(n_episodes=300, eps=eps, seed=3)
    smooth = np.convolve(lengths, np.ones(10) / 10, mode="valid")
    ax.plot(smooth, label=f"ε = {eps}  (optimal in {agreement(Q, OPT):.0%} of states)")
ax.set_xlabel("episode"); ax.set_ylabel("moves per episode (10-episode average)")
ax.set_yscale("log"); ax.legend(); plt.tight_layout(); plt.show()

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def step(s, a):
    r, c = to_rc(s)
    dr, dc = ACTIONS[a]
    nr, nc = r + dr, c + dc
    if not (0 <= nr < N_ROWS and 0 <= nc < N_COLS) or (nr, nc) == WALL:
        nr, nc = r, c
    s2 = to_s((nr, nc))
    if s2 == to_s(GOAL):
        return s2, 1.0, True
    if s2 == to_s(PIT):
        return s2, -1.0, True
    return s2, 0.0, False

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def value_iteration(gamma=GAMMA, tol=1e-10):
    V = np.zeros(16)
    sweeps = 0
    while True:
        V_new = V.copy()
        for s in NONTERMINAL:
            best = -np.inf
            for a in range(4):
                s2, r, done = step(s, a)
                best = max(best, r + (0.0 if done else gamma * V[s2]))
            V_new[s] = best
        delta = np.max(np.abs(V_new - V))
        V = V_new
        if delta < tol:
            break
        sweeps += 1
    return V, sweeps

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def epsilon_greedy(Q, s, eps, rng):
    if rng.random() < eps:
        return int(rng.integers(4))
    best = np.flatnonzero(Q[s] == Q[s].max())
    return int(rng.choice(best))

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def q_learning(n_episodes=500, alpha=0.5, gamma=GAMMA, eps=0.2, seed=0, max_moves=200):
    rng = np.random.default_rng(seed)
    Q = np.zeros((16, 4))
    lengths = []
    for ep in range(n_episodes):
        s, done, moves = to_s(START), False, 0
        while not done and moves < max_moves:
            a = epsilon_greedy(Q, s, eps, rng)
            s2, r, done = step(s, a)
            target = r + (0.0 if done else gamma * Q[s2].max())
            Q[s, a] += alpha * (target - Q[s, a])
            s, moves = s2, moves + 1
        lengths.append(moves)
    return Q, lengths

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def agreement(Q, opt):
    ok = 0
    for s in NONTERMINAL:
        greedy = set(np.flatnonzero(np.isclose(Q[s], Q[s].max())))
        ok += greedy <= opt[s]
    return ok / len(NONTERMINAL)